# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abood-arc/Flyrank-ml-project/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [12]:
print("""Binary classification. The target is a 0/1 label  is this page declining or not
— and yes/no is about as classification as it gets. Not clustering (I'm not looking for
groups, I already know what I'm predicting). Not really ranking either, even though the
end product is a ranked list: the model itself is trained to answer one yes/no question
per page, and the ranking is just what you get when you sort by how confident it is.
Same shape as the depth-2 tree in notebook 02 a classifier, used downstream for a queue.""")


Binary classification. The target is a 0/1 label  is this page declining or not
— and yes/no is about as classification as it gets. Not clustering (I'm not looking for
groups, I already know what I'm predicting). Not really ranking either, even though the
end product is a ranked list: the model itself is trained to answer one yes/no question
per page, and the ranking is just what you get when you sort by how confident it is.
Same shape as the depth-2 tree in notebook 02 a classifier, used downstream for a queue.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [11]:
print("""The label is a proxy, and a fairly involved one, so here's exactly how it's built
and why.

Naive version: 'declining' = trend_pct below some fixed cutoff, same for every page. I tried
that first and the data talked me out of it. Client decline rates in this snapshot range from
0% to 93.7%  some clients are just having a rough stretch account-wide, some are thriving.
A fixed global cutoff would flag almost every page for a struggling client and almost none for
a booming one, which isn't 'this page has a problem,' it's 'this client is having a quarter.'
Not the same thing, not equally actionable.

So the real label compares each page to its OWN client's typical trend_pct, not to the whole
dataset: relative_gap = page's trend_pct  that client's median trend_pct. A page is labeled
declining when it sits more than 20 points below its own client's median  20 landed there
because it's close to the natural 25th-percentile break in the relative-gap distribution, not
a round number I liked the look of.

Two things had to be handled explicitly, not swept under a fillna(0):
- Clients with too few usable pages (checked: one client had only 2) don't get a trustworthy
  median, so their pages are excluded, not silently guessed at.
- Pages with no real trend_pct at all (a whole client here has zero prior-window data — brand
  new, still ramping up tracking) are excluded too, not defaulted to 'not declining.' A missing
  answer is not the same as a no.

This is a proxy, not ground truth — trend_pct itself is a snapshot ratio, not a measured causal
outcome, and I can't yet check whether a flagged drop holds up over more than one window
(this starter file only has one before/after pair). That's a real gap, and closing it needs
the daily warehouse, not this CSV. client_id is used only to compute this label and for
grouped train/test splits  it never goes into the model as a feature.""")


The label is a proxy, and a fairly involved one, so here's exactly how it's built
and why.

Naive version: 'declining' = trend_pct below some fixed cutoff, same for every page. I tried
that first and the data talked me out of it. Client decline rates in this snapshot range from
0% to 93.7%  some clients are just having a rough stretch account-wide, some are thriving.
A fixed global cutoff would flag almost every page for a struggling client and almost none for
a booming one, which isn't 'this page has a problem,' it's 'this client is having a quarter.'
Not the same thing, not equally actionable.

So the real label compares each page to its OWN client's typical trend_pct, not to the whole
dataset: relative_gap = page's trend_pct  that client's median trend_pct. A page is labeled
declining when it sits more than 20 points below its own client's median  20 landed there
because it's close to the natural 25th-percentile break in the relative-gap distribution, not
a round number I liked the 

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [10]:
print("""Precision@50, primary. Of the top 50 pages the model ranks as most likely declining,
what fraction actually are? That's the number that maps onto the real action  an editor with
limited time working down a list, not a stats class exercise. It's also the same metric the
starter pipeline reports (baseline 0.240, random forest 0.740), so my number means something
next to theirs, not just in a vacuum.

Precision@20 comes along as a secondary check, because with only 24 clients contributing usable
labels, a client-holdout test fold could end up small, and I want to know the model still holds
up if a reviewer can only realistically get through 20 pages, not 50. Both get computed only
over rows eligible for a trustworthy label  the excluded ones don't get to quietly pad or
dent the score either way.""")


Precision@50, primary. Of the top 50 pages the model ranks as most likely declining,
what fraction actually are? That's the number that maps onto the real action  an editor with
limited time working down a list, not a stats class exercise. It's also the same metric the
starter pipeline reports (baseline 0.240, random forest 0.740), so my number means something
next to theirs, not just in a vacuum.

Precision@20 comes along as a secondary check, because with only 24 clients contributing usable
labels, a client-holdout test fold could end up small, and I want to know the model still holds
up if a reviewer can only realistically get through 20 pages, not 50. Both get computed only
over rows eligible for a trustworthy label  the excluded ones don't get to quietly pad or
dent the score either way.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [6]:
import pandas as pd, numpy as np
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Abood-arc/Flyrank-ml-project"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")  # work/notebooks -> work -> repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — check the path above"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Volume floor: drop pages with too little prior traffic to trust a % change
# (a page going from 1 impression to 450 isn't a "44,900% story," it's noise).
clean = df.loc[df["impressions_prev_30d"] >= 50].copy()

# Client-relative baseline, built only from this snapshot's feature-window data.
MIN_CLIENT_PAGES = 20
client_stats = clean.groupby("client_id")["trend_pct"].agg(["median", "count"])
client_stats.columns = ["client_median_trend", "client_usable_pages"]
clean = clean.merge(client_stats, on="client_id", how="left")
clean["relative_gap"] = clean["trend_pct"] - clean["client_median_trend"]

eligible = (
    clean["trend_pct"].notna()
    & clean["client_median_trend"].notna()
    & (clean["client_usable_pages"] >= MIN_CLIENT_PAGES)
)
clean["declining_relative"] = np.where(eligible, (clean["relative_gap"] < -20).astype(int), np.nan)

# One row = one content page, evaluated at this snapshot, labeled against its OWN client's norm.
unit_df = clean.loc[eligible, [
    "content_id", "client_id", "trend_pct", "client_median_trend",
    "relative_gap", "declining_relative", "content_type", "word_count"
]]

print(f"Eligible rows: {len(unit_df)} of {len(df)} total")
print(unit_df.head(8).to_string(index=False))
print()
print("Target distribution:")
print(unit_df["declining_relative"].value_counts())


Working dir: /content/flyrank-ml-internship-starter
Eligible rows: 20211 of 30000 total
          content_id         client_id  trend_pct  client_median_trend  relative_gap  declining_relative    content_type  word_count
content_304f48230142 client_f369cb89fc      -41.4               -41.40          0.00                 0.0 keyword article      3221.0
content_a1fb4e703a9e client_4e07408562      -57.7               -19.40        -38.30                 1.0 keyword article      2481.0
content_9aa793d4d895 client_7f2253d7e2      -60.9               -65.40          4.50                 0.0 keyword article      3515.0
content_331d6c4de07b client_19581e27de      -13.8               -20.00          6.20                 0.0 keyword article         NaN
content_d99b7a2d90ca client_3fdba35f04      -34.7               -59.10         24.40                 0.0 keyword article      2803.0
content_d4084a4bc775 client_f369cb89fc      -38.9               -41.40          2.50                 0.0 keyword a

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [9]:
print("""Look at the dataframe above. Two pages can have nearly the same raw trend_pct and
land on opposite sides of the label, because what counts as 'bad' depends on which client's
normal you're comparing against. A single if-statement can't hold that  you'd need a
different hardcoded cutoff per client, re-tuned every time a client's baseline drifts, which
stops being a rule and starts being 30-plus rules pretending to be one.

That's the actual argument for ML here, and I want to be honest about its size: this isn't
'the pattern is too complex for humans to ever see.' It's that the right comparison shifts per
client, and a model can learn 'compare each page to its own group's norm' as a general
relationship instead of me hand-writing and maintaining one cutoff per client by hand.""")


Look at the dataframe above. Two pages can have nearly the same raw trend_pct and
land on opposite sides of the label, because what counts as 'bad' depends on which client's
normal you're comparing against. A single if-statement can't hold that  you'd need a
different hardcoded cutoff per client, re-tuned every time a client's baseline drifts, which
stops being a rule and starts being 30-plus rules pretending to be one.

That's the actual argument for ML here, and I want to be honest about its size: this isn't
'the pattern is too complex for humans to ever see.' It's that the right comparison shifts per
client, and a model can learn 'compare each page to its own group's norm' as a general
relationship instead of me hand-writing and maintaining one cutoff per client by hand.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.